# Demo 2: Controlled Pendulum
## Demo 2.1: Hybrid - Unified within SysSimX Framework

### Description

The following demo implements a controlled pendulum system described in Demo 2.1. The Modelica models for the `Reference`, `AngleEncoder`, `Controller`, and `Drive` are exported as an Co-Simulation FMU using OpenModelica. The FMUs are then imported and simulated in Python using the FMPy package.

The pendulum itself again modeled as an OpenSim model, which is used via the OpenSim Python API within the Co-Simulation loop.

### Features of the `SysSimX` Framework

- The Framework supports the **configuration of FMUs within a YAML configuration file** where the path to the FMU file, inputs, and outputs can be specified
- The framework provides a **unified interface for working with co-simulation FMUs and OpenSim models**
- Both FMUs and OpenSim models follow the **CoSimComponent interface protocol**, allowing for seamless integration and interaction
- Similar methods for getting and setting variables, initialization and setup, and performing simulation steps


### Procedure

**1. Changing the working dirctory to use `SysSimX` package**

In [17]:
import os
import sys
from pathlib import Path
repo_root = Path.cwd().parent.parent
sys.path.insert(0, str(repo_root))

**2. Load the configuration for the involved FMUs**

In [18]:
from SysSimX.core.config import load_config

cfg_path = repo_root / 'SysSimX' / 'demos' / 'configs' / 'demo_hybrid.yaml'

# Check if the file exists
if not cfg_path.is_file():
    raise FileNotFoundError(f"The configuration file was not found: {cfg_path}")

cfg = load_config(str(cfg_path))

print("Configuration loaded successfully:")
for key, value in cfg.items():
    print(f"{key}: {value}")
    for subkey, subvalue in value.items():
        print(f"  {subkey}: {subvalue}")
        if isinstance(subvalue, dict):
            for subsubkey, subsubvalue in subvalue.items():
                print(f"    {subsubkey}: {subsubvalue}")

Configuration loaded successfully:
step: {'t0': 0.0, 'tf': 10.0, 'h': 0.001}
  t0: 0.0
  tf: 10.0
  h: 0.001
fmus: {'Reference': {'path': '/home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/Reference.fmu', 'inputs': {}, 'outputs': {'q_ref': 'q_ref'}}, 'SensorRef': {'path': '/home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/AngleEncoder.fmu', 'inputs': {'q': 'q'}, 'outputs': {'U_q': 'U_q'}}, 'SensorState': {'path': '/home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/AngleEncoder.fmu', 'inputs': {'q': 'q'}, 'outputs': {'U_q': 'U_q'}}, 'PID': {'path': '/home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/PID_Continuous.fmu', 'inputs': {'ref': 'ref', 'y': 'y'}, 'outputs': {'u': 'u'}}, 'Drive': {'path': '/home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/Drive.fmu', 'inputs': {'u_control': 'u_control', 'omega': 'omega'}, 'outputs': {'torque': 'torque', 'omega': 'omega'}}}
  Reference: {'path': '/home/flo/repos/SystemSimulation/demos

In [19]:
t0, tf, h = cfg["step"]["t0"], cfg["step"]["tf"], cfg["step"]["h"]

print(f"Simulation start time: {t0}")
print(f"Simulation end time: {tf}")
print(f"Simulation step size: {h}")

Simulation start time: 0.0
Simulation end time: 10.0
Simulation step size: 0.001


**3. Build the FMU Co-Simulation Components**

In [20]:
from SysSimX.components.fmu_component import FMUComponent

# Build FMU components
ref = FMUComponent('ref', cfg['fmus']['Reference']['path'],
                   inputs=cfg['fmus']['Reference']['inputs'],
                   outputs=cfg['fmus']['Reference']['outputs'])
sref  = FMUComponent("sref",  cfg["fmus"]["SensorRef"]["path"],
                    inputs=cfg["fmus"]["SensorRef"]["inputs"],
                    outputs=cfg["fmus"]["SensorRef"]["outputs"])
sstate= FMUComponent("sstate",cfg["fmus"]["SensorState"]["path"],
                    inputs=cfg["fmus"]["SensorState"]["inputs"],
                    outputs=cfg["fmus"]["SensorState"]["outputs"])
pid   = FMUComponent("pid",   cfg["fmus"]["PID"]["path"],
                    inputs=cfg["fmus"]["PID"]["inputs"],
                    outputs=cfg["fmus"]["PID"]["outputs"])
drive = FMUComponent("drive", cfg["fmus"]["Drive"]["path"],
                    inputs=cfg["fmus"]["Drive"]["inputs"],
                    outputs=cfg["fmus"]["Drive"]["outputs"])

**4. Build the OpenSim Pendulum Component**

In [21]:
from SysSimX.components.opensim_pendulum import OpenSimPendulum

plant = OpenSimPendulum()

**5. Initialize all components**

In [22]:
for component in [ref, sref, sstate, pid, drive, plant]:
    component.initialize(t0)

**6. Run the Co-Simulation with Logger**

In [23]:
from SysSimX.core.scheduler import run_controlled_pendulum

# Logger
rows = []
def log(t, signals):
    rows.append({"t": t, **signals})

run_controlled_pendulum(ref=ref, sensor_ref=sref, sensor_state=sstate,
                        pid=pid, drive=drive, plant=plant,
                        t0=t0, tf=tf, h=h, logger=log)

print(f"Simulation completed with {len(rows)} steps.")
print("Final state values:")
for key, value in rows[-1].items():
    print(f"{key:<11}: {value:.4f}")

Simulation completed with 10000 steps.
Final state values:
t          : 9.9990
q_ref      : 0.0000
q_state    : 0.1597
omega_state: -1.1362
U_q_ref    : 1.4941
U_q_state  : 1.6588
u_pid      : -0.0284
torque     : 24.8262


**7. Plot the results**

In [24]:
import numpy as np
t_vals = [row['t'] for row in rows]
q_ref_vals = [np.rad2deg(row['q_ref']) for row in rows]
q_state_vals = [np.rad2deg(row['q_state']) for row in rows]

# Create a simple plotly figure for the q_ref and q_state over time
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_vals, y=q_ref_vals, mode='lines',
                            name='Reference Angle (q_ref)', line=dict(color='white', dash='dash')))
fig.add_trace(go.Scatter(x=t_vals, y=q_state_vals, mode='lines',
                            name='State Angle (q_state)', line=dict(color='red')))
fig.update_layout(title='Pendulum Angle Tracking',
                  xaxis_title='Time (s)',
                  yaxis_title='Angle (degrees)',
                  legend_title='Legend',
                  template='plotly_dark')
fig.show()